In [1]:
import numpy as np

# Way 1: From Scratch

In [12]:
# Data 
# Features: hours, studied, hours slept, practice tests taken

X_raw = np.array([
    [2, 7, 1],
    [3, 6, 2],
    [5, 8, 3],
    [6, 5, 2],
    [7, 7, 4],
    [8, 6, 5]
])

y = np.array([50, 58, 72, 68, 85, 90])

# Add bias column (column of 1s for theta_0)
n = X_raw.shape[0]
X = np.hstack([np.ones((n, 1)), X_raw]) # shape: (6, 4)


# Calculate theta
theta = np.linalg.inv(X.T @ X) @ X.T @ y

print("=" * 45)
print(f"  Intercept (θ₀)        : {theta[0]:.2f}")
print(f"  Hours coef (θ₁)       : {theta[1]:.2f}")
print(f"  Sleep coef (θ₂)       : {theta[2]:.2f}")
print(f"  Practice coef (θ₃)    : {theta[3]:.2f}")
print("=" * 45)

# Predict
def predict(x_new):
    x_with_bias = np.hstack([1, x_new])
    return x_with_bias @ theta

# Predict for: 6 hours study, 7 hours sleep, 3 practice tests
print(f"\n  Prediction: {predict([6, 7, 3]):.2f}")

  Intercept (θ₀)        : 28.59
  Hours coef (θ₁)       : 4.01
  Sleep coef (θ₂)       : 1.33
  Practice coef (θ₃)    : 4.43

  Prediction: 75.24


# Way 2: Using Scikit-Learn

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [19]:
# Data
X = np.array([
    [2, 7, 1],
    [3, 6, 2],
    [5, 8, 3],
    [6, 5, 2],
    [7, 7, 4],
    [8, 6, 5]
])

y = np.array([50, 58, 72, 68, 85, 90])

# Train/Test Split
# Always split your data - never evaluate on trainig data alone
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Train
model = LinearRegression()
model.fit(X_train, y_train)

# Parameters
print(f"Intercept: {model.intercept_:.2f}")
print(f"Coefs: {model.coef_}")

# Evaluate
train_r2 = r2_score(y_train, model.predict(X_train))
test_r2 = r2_score(y_test, model.predict(X_test))

# Predict new student
new_student = np.array([[6, 7, 3]])  # hours, sleep, practice
print(f"\n  Prediction: {model.predict(new_student)[0]:.2f}")

Intercept: -0.25
Coefs: [8.   3.75 0.75]

  Prediction: 76.25


In [17]:
model.predict(X_train), model.predict(X_test)


(array([90., 72., 85., 68.]), array([42.75, 47.75]))

# Checking Multicollinearity --- VIF

In [21]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [23]:
# VIF needs a Dataframe
X_df = pd.DataFrame(X, columns = ['hours', 'sleep', 'practice'])

vif_data = pd.DataFrame({
    'Feature': X_df.columns,
    'VIF': [variance_inflation_factor(X_df.values, i) for i in range(X_df.shape[1])]
})

print(vif_data)

    Feature        VIF
0     hours  28.148902
1     sleep   5.584622
2  practice  26.499593


# 📘 Session 02 — Multiple Linear Regression
### *Data Science & Machine Learning — Personal Study Notes*
---
> **"The model's goal is not to memorize your data — it's to learn the pattern well enough to predict data it has never seen before."**

---

## 📌 Table of Contents
1. [What Is Multiple Linear Regression?](#1-what-is-multiple-linear-regression)
2. [The Math Behind It](#2-the-math-behind-it)
3. [The 5 Critical Assumptions](#3-the-5-critical-assumptions)
4. [R² vs Adjusted R²](#4-r-vs-adjusted-r)
5. [Train/Test Split](#5-traintest-split)
6. [What model.predict() Actually Does](#6-what-modelpredict-actually-does)
7. [Do Predictions Match Actual Values?](#7-do-predictions-match-actual-values)
8. [Multicollinearity — The Hidden Enemy](#8-multicollinearity--the-hidden-enemy)
9. [VIF — Variance Inflation Factor](#9-vif--variance-inflation-factor)
10. [Failure Cases](#10-failure-cases)
11. [Key Concepts From Questions Asked](#11-key-concepts-from-questions-asked)
12. [Summary Cheatsheet](#12-summary-cheatsheet)

---

## 1. What Is Multiple Linear Regression?

**Simple Linear Regression** used one input to predict one output.
**Multiple Linear Regression** uses **two or more inputs** to predict one output — simultaneously.

### Intuition First

In Session 1, you predicted exam scores from hours studied alone. But reality is richer:

- Hours slept the night before
- Number of practice tests taken
- Attendance percentage

All of these affect the score. Multiple Linear Regression considers **all relevant factors at the same time** to make a better, more honest prediction.

> ✅ **One input → Simple Linear Regression**
> ✅ **Two or more inputs → Multiple Linear Regression**
> The core idea is identical. Only the scale changes.

---

## 2. The Math Behind It

### The Equation

Simple regression:
$$\hat{y} = \theta_0 + \theta_1 x$$

Multiple regression with **n features:**
$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3 + \cdots + \theta_n x_n$$

For our example:
$$\text{score} = \theta_0 + \theta_1(\text{hours}) + \theta_2(\text{sleep}) + \theta_3(\text{practice tests})$$

> 💡 Each θ means: *"Holding everything else constant, how much does y change when this one feature increases by 1?"*

---

### What Each Coefficient Actually Means

Suppose your trained model gives:
$$\text{score} = 20 + 6.5(\text{hours}) + 3.1(\text{sleep}) + 4.8(\text{practice tests})$$

| Coefficient | Value | Meaning |
|---|---|---|
| θ₀ | 20 | Base score assuming all features are zero |
| θ₁ (hours) | 6.5 | Each extra study hour adds **6.5 points** — *if sleep and practice stay the same* |
| θ₂ (sleep) | 3.1 | Each extra sleep hour adds **3.1 points** — *if hours and practice stay the same* |
| θ₃ (practice) | 4.8 | Each extra practice test adds **4.8 points** — *if hours and sleep stay the same* |

> ⚠️ **"Holding everything else constant"** is the most important phrase in multiple regression. Never forget it when interpreting coefficients.

---

### Matrix Form — Your First Real Linear Algebra Moment

Writing out every θ separately becomes impossible with 50+ features. So we use **matrix notation:**

$$\hat{y} = X\theta$$

Where:

$$X = \begin{bmatrix} 1 & x_1^{(1)} & x_2^{(1)} \\ 1 & x_1^{(2)} & x_2^{(2)} \\ 1 & x_1^{(3)} & x_2^{(3)} \end{bmatrix}, \quad \theta = \begin{bmatrix} \theta_0 \\ \theta_1 \\ \theta_2 \end{bmatrix}$$

The OLS closed-form solution in matrix form:
$$\theta = (X^T X)^{-1} X^T y$$

> Don't memorize this formula yet. Understand what it means: *"Find the exact θ values that minimize MSE across all features simultaneously."*
> This is exactly **why linear algebra exists in ML** — not as abstract math, but as a tool that makes large-scale problems elegantly solvable.

---

### Why We Add a Column of 1s (Bias Column)

The equation has a θ₀ that needs something to multiply with in the matrix. We give it **1** — because anything × 1 = itself:

$$\hat{y} = \begin{bmatrix} 1 & x_1 & x_2 & x_3 \end{bmatrix} \begin{bmatrix} \theta_0 \\ \theta_1 \\ \theta_2 \\ \theta_3 \end{bmatrix} = \theta_0 \cdot 1 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3$$

> The column of 1s is a **mathematical trick** that lets θ₀ participate in matrix multiplication cleanly. Without it, the matrix equation simply doesn't work.

---

### Understanding `.shape`, `hstack`, and Transpose `.T`

These three tools appear in the from-scratch code. Here's exactly what each one does:

#### `.shape`
Tells you the **dimensions** of an array — returns `(rows, columns)`:
- `X.shape[0]` → number of rows (number of students/samples)
- `X.shape[1]` → number of columns (number of features)

We use `shape[0]` to know **how many 1s to create** for the bias column — one per sample.

#### `hstack` — Horizontal Stack
Glues arrays together **side by side**, adding columns:

```
Bias column    +    Features    =    Final X
[[1.]               [[2, 7, 1],      [[1, 2, 7, 1],
 [1.]                [3, 6, 2],       [1, 3, 6, 2],
 [1.]    hstack →    [5, 8, 3],  →    [1, 5, 8, 3],
 [1.]                [6, 5, 2],       [1, 6, 5, 2],
 [1.]                [7, 7, 4],       [1, 7, 7, 4],
 [1.]]               [8, 6, 5]]       [1, 8, 6, 5]]
```

#### `.T` — Transpose
**Flips a matrix** so rows become columns and columns become rows.

If X is shape **(6 × 4)** → X.T is shape **(4 × 6)**

Why we need it in $\theta = (X^TX)^{-1}X^Ty$:

| Part | Shape Math | Why |
|---|---|---|
| $X^T X$ | (4×6) × (6×4) = **(4×4)** | Shapes must align for multiplication |
| $X^T y$ | (4×6) × (6,) = **(4,)** | Gives us one θ value per feature |

> Without transpose, matrix shapes simply don't align and multiplication is impossible.

---

## 3. The 5 Critical Assumptions

A model can look great on the surface while being completely invalid underneath. These assumptions must be met for your results to be trustworthy:

| # | Assumption | Plain English | How to Check |
|---|---|---|---|
| 1 | **Linearity** | Each feature has a straight-line relationship with y | Scatter plots of each feature vs y |
| 2 | **Independence** | One observation doesn't influence another | Think about your data collection process |
| 3 | **Homoscedasticity** | Errors have constant spread across all predictions | Residual vs fitted values plot — should look random |
| 4 | **Normality of errors** | Residuals follow a normal distribution | Histogram of residuals, Q-Q plot |
| 5 | **No Multicollinearity** | Features are not highly correlated with *each other* | Correlation matrix, VIF scores |

> ⚠️ **Assumption 5 is the dangerous one.** It is unique to multiple regression and easy to miss. See Section 8 for full details.

---

## 4. R² vs Adjusted R²

### What R² Does in Multiple Regression

R² means the same thing it always does:
> *"How much of the variation in y do all my features combined explain?"*

If R² = 0.91 → hours studied, sleep, and practice tests **together** explain 91% of score variation.

---

### The Flaw — R² Never Goes Down

**Every time you add a new feature, R² increases — even if that feature is completely useless.**

| Model | Features Added | R² | Adjusted R² |
|---|---|---|---|
| Model 1 | hours | 0.780 | 0.776 |
| Model 2 | + sleep | 0.850 | 0.844 ✅ |
| Model 3 | + practice | 0.910 | 0.903 ✅ |
| Model 4 | + shoe size 👟 | 0.913 | 0.901 ⬇️ |
| Model 5 | + favorite color 🎨 | 0.915 | 0.898 ⬇️ |
| Model 6 | + random noise | 0.916 | 0.895 ⬇️ |

R² went up every single time — even for shoe size, favorite color, and random noise.
Adjusted R² went up only when features were **genuinely useful.**

---

### Why This Happens

Every new feature gives the model a tiny extra degree of freedom to fit the training data slightly better — even random noise accidentally aligns with data a little bit. Plain R² sees that tiny improvement and counts it as progress.

---

### Adjusted R² — The Honest Version

Adjusted R² penalizes you for every feature added. A new feature must **earn its place.**

$$\bar{R}^2 = 1 - \frac{(1-R^2)(n-1)}{n-p-1}$$

| Symbol | Meaning |
|---|---|
| $n$ | Number of samples |
| $p$ | Number of features |
| $(n-p-1)$ | Gets smaller as p increases → penalty gets larger |

As you add more features → penalty grows → Adjusted R² gets pulled down — **unless the feature genuinely helps.**

---

### The Core Difference

| | R² | Adjusted R² |
|---|---|---|
| **When feature is useful** | Goes up ✅ | Goes up ✅ |
| **When feature is useless** | Still goes up ⚠️ | Goes down ✅ |
| **Can it decrease?** | Never | Yes — by design |
| **Use in simple regression** | ✅ Fine | Same result |
| **Use in multiple regression** | ⚠️ Misleading | ✅ Always use this |

> **R² is an optimist — it celebrates every new feature.**
> **Adjusted R² is an honest critic — it only celebrates features that genuinely earn their place.**

---

## 5. Train/Test Split

### Why We Split

Evaluating your model on the same data it trained on is like giving a student the exact exam questions during practice — of course they'll score 100%. It tells you nothing about real performance.

| Split | Typical Size | Purpose |
|---|---|---|
| **Training set** | 80% | Model learns from this |
| **Test set** | 20% | Model is evaluated on this — **never seen during training** |

> ✅ **The test R² is the only honest score.**
> Train R² tells you the model showed up to practice.
> Test R² tells you if it can actually play the game.

---

### Reading the Gap Between Train and Test R²

| Train R² | Test R² | Diagnosis |
|---|---|---|
| 0.91 | 0.89 | ✅ Healthy — model generalizes well |
| 0.94 | 0.51 | ❌ Overfitting — memorized training data |
| 0.60 | 0.58 | ⚠️ Underfitting — model too simple for the data |

---

## 6. What `model.predict()` Actually Does

Nothing magical. It runs the stored equation on every row of input data:

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3$$

For a student with `[hours=5, sleep=8, practice=3]`:
$$\hat{y} = 20 + 6.5(5) + 3.1(8) + 4.8(3) = 20 + 32.5 + 24.8 + 14.4 = 91.7$$

It does this for **every single row simultaneously** using matrix multiplication: $\hat{Y} = X\theta$

### Why We Predict on Both Train and Test

| Prediction | Question Being Asked |
|---|---|
| `model.predict(X_train)` | *"How well did the model learn what it was trained on?"* |
| `model.predict(X_test)` | *"How well does the model perform on data it has never seen?"* |

The **gap between them** is the diagnostic tool. See Section 5.

---

## 7. Do Predictions Match Actual Values?

**Not exactly — and that is perfectly normal.**

The model doesn't memorize your data. It finds the **single best fit across all points simultaneously.** No one point gets a perfect prediction — every point gets a small error so the overall MSE across all points is minimized.

Think of it visually:
```
Actual points:      ●  ●  ●  ●  ●  ●
Best fit line:   ──────────────────────
                  ↑some points    ↑some points
                  above line      below line
```

Those small differences between actual and predicted values are the **residuals** — the raw material of MSE.

### When Would Predictions Match Exactly?

| Situation | Result |
|---|---|
| R² = 1.0 | Every prediction matches exactly |
| R² < 1.0 | Close but not exact — normal and healthy |
| Train R² = 1.0 with many features | ⚠️ Danger — likely overfitting |

> ✅ Perfect predictions on training data is a **warning sign**, not a success.

### Comparing Actual vs Predicted — The Residuals Table

Always build this comparison after training:

| Actual | Predicted | Error (Residual) |
|---|---|---|
| 50 | 51.3 | -1.3 |
| 58 | 56.8 | 1.2 |
| 72 | 73.1 | -1.1 |
| 68 | 67.4 | 0.6 |
| 85 | 84.2 | 0.8 |
| 90 | 91.2 | -1.2 |

Small, random, scattered residuals = healthy model.
Large or patterned residuals = something is wrong.

---

## 8. Multicollinearity — The Hidden Enemy

### What Is It?

Multicollinearity happens when two or more **input features** are highly correlated with each other — they carry the same information.

**Example:** Including both *hours studied* and *pages read*. A student who studies more also reads more pages — these two features move together almost perfectly. The model gets confused about which one is actually causing the score to improve.

### Why It's Dangerous

- Coefficients become **unstable** — a tiny change in data wildly shifts θ values
- Coefficients become **uninterpretable** — you can no longer trust what they mean
- Your R² can look **perfectly fine** — making this a silent killer

> ⚠️ **This is one of the most dangerous bugs in ML** — your model looks healthy on the surface while completely broken underneath.

### A Real Example

```
Features: temperature + heat_index
```
Heat index is mathematically calculated from temperature. They are almost identical information. The model cannot separate their individual effects on y — coefficients become meaningless.

### How to Fix It

| Fix | When To Use |
|---|---|
| **Drop one feature** | When two features carry the same information |
| **Combine them** | Average them or create a single ratio |
| **Ridge Regression** | Keep all features but reduce their impact — covered later |

---

## 9. VIF — Variance Inflation Factor

VIF is the tool that **measures multicollinearity** — it gives each feature a score showing how much that feature is explained by the other features.

### The Intuition

> VIF asks for each feature: *"Can this feature be predicted by the other features?"*
> If yes → the model is confused.
> If no → the model is safe.

### How VIF Is Calculated

For each feature, VIF runs a **brand new regression** using that feature as the target y and all other features as inputs:

| Step | What Happens |
|---|---|
| VIF for *hours* | Regress hours on [sleep, practice] → get R² |
| VIF for *sleep* | Regress sleep on [hours, practice] → get R² |
| VIF for *practice* | Regress practice on [hours, sleep] → get R² |

Then the formula:
$$\text{VIF}_i = \frac{1}{1 - R_i^2}$$

### What The Formula Is Saying

| R² from regression | Meaning | VIF Result |
|---|---|---|
| R² = 0 | Zero relationship with other features | VIF = 1 → perfectly safe |
| R² = 0.80 | Other features explain 80% of this feature | VIF = 5 → moderate concern |
| R² = 0.90 | Other features explain 90% of this feature | VIF = 10 → severe problem |
| R² = 0.99 | Other features explain 99% of this feature | VIF = 100 → critical |

> As R² approaches 1, the denominator $(1 - R^2)$ approaches zero. Dividing by something near zero gives a massive number — **that large number is the warning signal.**

### The Interpretation Scale

| VIF Value | Meaning | Action |
|---|---|---|
| **1** | No multicollinearity | ✅ Safe |
| **1 – 5** | Low — acceptable | ✅ Safe |
| **5 – 10** | Moderate | ⚠️ Investigate |
| **> 10** | Severe multicollinearity | ❌ Take action |

### A Real VIF Output Example

| Feature | VIF Score | Verdict |
|---|---|---|
| hours_studied | 1.2 | ✅ Safe |
| hours_slept | 1.8 | ✅ Safe |
| temperature | 54.3 | ❌ Severe |
| heat_index | 61.7 | ❌ Severe |

Temperature and heat index are mathematically related — VIF catches it immediately.

---

## 10. Failure Cases

| Failure Case | What Happens | How to Detect |
|---|---|---|
| **Multicollinearity** | Unstable, uninterpretable coefficients | Correlation matrix, VIF > 10 |
| **Irrelevant features** | Noise gets fitted, generalizes poorly | Check p-values, use feature selection |
| **Overfitting** | Memorizes training data, fails on new data | Train R² high, Test R² low |
| **Violated assumptions** | Entire model output becomes invalid | Always check all 5 assumptions |
| **Extrapolation** | Unreliable predictions outside training range | Never predict far beyond your data |

---

## 11. Key Concepts From Questions Asked

### Why `coef_` Has Multiple Values Now
In simple regression: `model.coef_` → `array([7.39])` — one slope for one feature.
In multiple regression: `model.coef_` → `array([6.5, 3.1, 4.8])` — one slope **per feature.**
This is exactly why `coef_` was always an array — it was designed for multiple features from the start.

### `shape[0]` vs `shape[1]`
- `shape[0]` → rows → number of samples
- `shape[1]` → columns → number of features
- Always use `shape[0]` when you need to match rows (e.g. creating the bias column)

### `hstack` vs `vstack`
- `hstack` → horizontal → adds **columns** (side by side)
- `vstack` → vertical → adds **rows** (on top of each other)
- We used `hstack` to attach the bias column to the left of our feature matrix

### Transpose Rules in Matrix Multiplication
- Matrix shapes must be compatible: inner dimensions must match
- X is (6×4) → X.T is (4×6)
- $(X^TX)$ = (4×6)(6×4) = (4×4) ✅
- $(X^Ty)$ = (4×6)(6,) = (4,) ✅
- Without transpose → shapes don't align → multiplication fails

---

## 12. Summary Cheatsheet

| Concept | One-Line Definition |
|---|---|
| **Multiple Linear Regression** | Best fit through data using two or more features simultaneously |
| **Each θ coefficient** | Effect of one feature on y, holding all other features constant |
| **Matrix form $X\theta$** | Compact notation that handles any number of features elegantly |
| **Bias column of 1s** | Mathematical trick so θ₀ participates in matrix multiplication |
| **Transpose `.T`** | Flips matrix rows and columns so shapes align for multiplication |
| **hstack** | Glues arrays side by side — adds columns |
| **5 Assumptions** | Must be met for regression results to be valid |
| **Train/Test Split** | Evaluate on unseen data — the only honest performance measure |
| **R²** | How much variation in y all features explain — never decreases with more features |
| **Adjusted R²** | Penalizes useless features — always use this in multiple regression |
| **Residual** | Difference between actual and predicted value — $y - \hat{y}$ |
| **Multicollinearity** | Features correlated with each other — makes coefficients untrustworthy |
| **VIF** | Measures multicollinearity — score above 10 means take action |
| **Overfitting** | High train R², low test R² — model memorized instead of learning |

### The 4-Step Learning Framework (Every Session)
```
1. INTUITION    → What is it trying to do?
2. MATH         → Why does the formula look like that?
3. CODE         → How do we build and use it?
4. FAILURE      → When does it break and why?
```

---

## 📝 Homework

- [ ] Run from-scratch code and print the θ values
- [ ] Run sklearn code and compare Train R² vs Test R²
- [ ] Build the Actual vs Predicted comparison table
- [ ] Run VIF on the features and interpret the scores
- [ ] Answer: *You have features — temperature, humidity, and heat index (calculated from temperature and humidity). What problem does this cause and what should you do?*
- [ ] Answer: *Your model shows Train R² = 0.94 but Test R² = 0.51. What is happening and why?*
- [ ] Think about: *Why does $\theta = (X^TX)^{-1}X^Ty$ fail when two features are perfectly correlated?*

---

*Next Session → **Logistic Regression** — same family, completely different purpose. Predicting categories instead of numbers.*

---
*Session 02 | Multiple Linear Regression | Personal DS/ML Study Notes*
